In [1]:
!pip install backtesting pandas-ta


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import sys
!{sys.executable} -m pip install ccxt

In [39]:
# pulling 10 day data for testing of strat 

import ccxt
import pandas as pd
from datetime import datetime
from backtesting import Backtest, Strategy

# --- SETTINGS ---
symbol = 'BCH/USDT'
timeframe = '5m'
start_date_str = "2026-03-07 00:00:00" # Choose your start
end_date_str   = "2026-03-17 00:00:00" # Choose your end (10 days later)

# 1. FETCH DATA FROM BINANCE
exchange = ccxt.binance()

# Convert strings to milliseconds for Binance
since = exchange.parse8601(start_date_str.replace(" ", "T") + "Z")
end_ts = exchange.parse8601(end_date_str.replace(" ", "T") + "Z")

all_ohlcv = []
while since < end_ts:
    ohlcv = exchange.fetch_ohlcv(symbol, timeframe, since, limit=1000)
    if not ohlcv:
        break
    
    # Filter out any data that might go beyond our end date
    last_ts = ohlcv[-1][0]
    if last_ts >= end_ts:
        # Only keep the ones before the end_ts
        ohlcv = [candle for candle in ohlcv if candle[0] < end_ts]
        all_ohlcv.extend(ohlcv)
        break
        
    all_ohlcv.extend(ohlcv)
    since = last_ts + 1 

# 2. FORMAT DATA FOR BACKTESTING.PY
data = pd.DataFrame(all_ohlcv, columns=['Timestamp', 'Open', 'High', 'Low', 'Close', 'Volume'])
data['Timestamp'] = pd.to_datetime(data['Timestamp'], unit='ms')
data.set_index('Timestamp', inplace=True)

# Ensure column names match Backtesting.py requirements
data = data[['Open', 'High', 'Low', 'Close', 'Volume']]

print(f"Downloaded {len(data)} bars from Binance between {start_date_str} and {end_date_str}.")
data.head()

Downloaded 2880 bars from Binance between 2026-03-07 00:00:00 and 2026-03-17 00:00:00.


,Open,High,Low,Close,Volume
Timestamp,,,,,
2026-03-07 00:00:00,449.6,450.8,449.6,450.5,31.107
2026-03-07 00:05:00,450.5,451.4,450.5,450.9,20.825
2026-03-07 00:10:00,450.9,451.1,450.6,450.6,41.653
2026-03-07 00:15:00,450.6,450.9,450.4,450.9,17.627
2026-03-07 00:20:00,451.0,451.0,449.8,450.4,28.835


In [40]:
# original percentile strat ever since moving to non parametric, initial model

class PercentileStrategy(Strategy):
    window = 160
    # Entry at the bottom 1% of the last 10 hours (120 * 5m)
    buy_quantile = 0.01  
    # Exit at the median (50th percentile)
    exit_quantile = 0.7

    def init(self):
        close = pd.Series(self.data.Close)
        
        # Calculate the rolling 1% "Floor"
        self.lower_bound = self.I(lambda x: x.rolling(self.window).quantile(self.buy_quantile), close)
        
        # Calculate the rolling 70% 
        self.median = self.I(lambda x: x.rolling(self.window).quantile(self.exit_quantile), close)

    def next(self):
        if len(self.data) < self.window:
            return

        price = self.data.Close[-1]
        
        # ENTRY: Only buy if price is in the literal bottom 1% of the window
        if price <= self.lower_bound[-1] and not self.position:
            self.buy(size=0.9)
            
        # EXIT: Sell once it returns to the median
        elif price >= self.median[-1] and self.position:
            self.position.close()

bt = Backtest(data, PercentileStrategy, cash=10000000, commission=0.001)
stats = bt.run()
print(stats)
print(stats['_trades']) # This will show you the log of every trade made

Backtest.run:   0%|          | 0/2720 [00:00<?, ?bar/s]

Start                     2026-03-07 00:00:00
End                       2026-03-16 23:55:00
Duration                      9 days 23:55:00
Exposure Time [%]                    28.05556
Equity Final [$]                10690871.8291
Equity Peak [$]                 10700513.4717
Commissions [$]                   205173.0709
Return [%]                            6.90872
Buy & Hold Return [%]                 6.51835
Return (Ann.) [%]                   544.56435
Volatility (Ann.) [%]                152.2325
CAGR [%]                           1046.42808
Sharpe Ratio                          3.57719
Sortino Ratio                       160.21123
Calmar Ratio                        255.93593
Alpha [%]                             5.67737
Beta                                  0.18891
Max. Drawdown [%]                    -2.12774
Avg. Drawdown [%]                    -0.23213
Max. Drawdown Duration        2 days 05:15:00
Avg. Drawdown Duration        0 days 04:15:00
# Trades                          

In [41]:
# show plots of performance

bt.plot()

GridPlot(id='p7136', ...)

In [36]:
# pulling data for testing 10 day windows with 5m timeframe since start of year

import ccxt
import pandas as pd
from datetime import datetime

# --- SETTINGS ---
symbol = 'BCH/USDT'
timeframe = '5m'
start_year = "2025-12-25 00:00:00"
end_now = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")

exchange = ccxt.binance()
since = exchange.parse8601(start_year.replace(" ", "T") + "Z")
end_ts = exchange.parse8601(end_now.replace(" ", "T") + "Z")

all_ohlcv = []
while since < end_ts:
    ohlcv = exchange.fetch_ohlcv(symbol, timeframe, since, limit=1000)
    if not ohlcv: break
    all_ohlcv.extend(ohlcv)
    since = ohlcv[-1][0] + 1
    if len(ohlcv) < 1000: break # Safety break

full_data = pd.DataFrame(all_ohlcv, columns=['Timestamp', 'Open', 'High', 'Low', 'Close', 'Volume'])
full_data['Timestamp'] = pd.to_datetime(full_data['Timestamp'], unit='ms')
full_data.set_index('Timestamp', inplace=True)
full_data = full_data[['Open', 'High', 'Low', 'Close', 'Volume']]

full_data = full_data.loc[:'2026-03-18 00:00:00'].copy()
full_data = full_data[~full_data.index.duplicated(keep='first')]
full_data = full_data.dropna()

print(f"Total Bars Loaded: {len(full_data)}")

Total Bars Loaded: 23905


In [6]:
# testing use independent 10 days window (data from start of the year) of the PercentileStrategy not Walk Forward Analysis

import matplotlib.pyplot as plt

# --- CONFIG ---
days_per_window = 10
bars_per_day = (24 * 60 // 5)  # 288 bars for 5m data
window_size = bars_per_day * days_per_window 

all_stats = []

# Loop BACKWARDS: Start at the end, jump back by window_size
# We use a negative step in range()
for i in range(len(full_data) - window_size, -1, -window_size):
    window_df = full_data.iloc[i : i + window_size]
    
    # Label by Start and End date for clarity
    start_label = window_df.index[0].strftime('%Y-%m-%d')
    end_label = window_df.index[-1].strftime('%m-%d')
    label = f"{start_label} to {end_label}"
    
    # Run Backtest
    bt = Backtest(window_df, PercentileStrategy, cash=1000000, commission=0.001)
    stats = bt.run()
    print(stats)
    print(stats['_trades'])
    
    # Store key metrics
    all_stats.append({
        'Window': label,
        'Return [%]': stats['Return [%]'],
        'Sharpe': stats['Sharpe Ratio'],
        'Sortino': stats['Sortino Ratio'],
        'Max DD [%]': stats['Max. Drawdown [%]'],
        'Win Rate [%]': stats['Win Rate [%]'],
        'Profit Factor': stats['Profit Factor'],
        'Trades': stats['# Trades']
    })

# Convert to DataFrame
results_df = pd.DataFrame(all_stats).set_index('Window').fillna(0)

# Display raw table in descending order (Newest first)
print("--- Backtest Results (Reverse Chronological) ---")
print(results_df)

Backtest.run:   0%|          | 0/2720 [00:00<?, ?bar/s]

Start                     2026-03-08 00:05:00
End                       2026-03-18 00:00:00
Duration                      9 days 23:55:00
Exposure Time [%]                    29.23611
Equity Final [$]                 1051466.0423
Equity Peak [$]                  1051752.4941
Commissions [$]                    18517.4329
Return [%]                             5.1466
Buy & Hold Return [%]                 5.64516
Return (Ann.) [%]                   575.18616
Volatility (Ann.) [%]               147.42295
CAGR [%]                            524.88917
Sharpe Ratio                          3.90161
Sortino Ratio                       205.27847
Calmar Ratio                        270.44029
Alpha [%]                             4.07659
Beta                                  0.18955
Max. Drawdown [%]                    -2.12685
Avg. Drawdown [%]                    -0.27627
Max. Drawdown Duration        2 days 05:15:00
Avg. Drawdown Duration        0 days 05:13:00
# Trades                          

/var/folders/f_/xv2kgn592hg5q7v5tl2sgx2m0000gn/T/ipykernel_59944/2982297518.py:22: UserWarning: Some trades remain open at the end of backtest. Use `Backtest(..., finalize_trades=True)` to close them and include them in stats.
  stats = bt.run()


Backtest.run:   0%|          | 0/2720 [00:00<?, ?bar/s]

Start                     2026-02-26 00:05:00
End                       2026-03-08 00:00:00
Duration                      9 days 23:55:00
Exposure Time [%]                    41.45833
Equity Final [$]                  996612.3495
Equity Peak [$]                  1002931.7152
Commissions [$]                    19145.3217
Return [%]                           -0.33877
Buy & Hold Return [%]                -9.64508
Return (Ann.) [%]                   121.44611
Volatility (Ann.) [%]                88.32142
CAGR [%]                            -11.65337
Sharpe Ratio                          1.37505
Sortino Ratio                         5.48994
Calmar Ratio                         12.10556
Alpha [%]                             3.17783
Beta                                   0.3646
Max. Drawdown [%]                   -10.03226
Avg. Drawdown [%]                    -5.05256
Max. Drawdown Duration        9 days 09:40:00
Avg. Drawdown Duration        4 days 16:55:00
# Trades                          

...

Start                     2026-01-17 00:05:00
End                       2026-01-27 00:00:00
Duration                      9 days 23:55:00
Exposure Time [%]                    38.95833
Equity Final [$]                 1011222.0621
Equity Peak [$]                  1028340.7686
Commissions [$]                    19993.0379
Return [%]                            1.12221
Buy & Hold Return [%]                -3.21608
Return (Ann.) [%]                    40.46235
Volatility (Ann.) [%]                41.60317
CAGR [%]                             50.30036
Sharpe Ratio                          0.97258
Sortino Ratio                         1.83845
Calmar Ratio                          8.46627
Alpha [%]                             2.39897
Beta                                  0.39699
Max. Drawdown [%]                    -4.77924
Avg. Drawdown [%]                    -0.67027
Max. Drawdown Duration        4 days 09:05:00
Avg. Drawdown Duration        0 days 09:01:00
# Trades                          

Backtest.run:   0%|          | 0/2720 [00:00<?, ?bar/s]

Start                     2026-01-07 00:05:00
End                       2026-01-17 00:00:00
Duration                      9 days 23:55:00
Exposure Time [%]                    31.21528
Equity Final [$]                 1024512.4293
Equity Peak [$]                  1051522.7306
Commissions [$]                    20174.4707
Return [%]                            2.45124
Buy & Hold Return [%]                -5.21492
Return (Ann.) [%]                    27.56493
Volatility (Ann.) [%]                22.98909
CAGR [%]                            142.10976
Sharpe Ratio                          1.19904
Sortino Ratio                          2.0478
Calmar Ratio                          5.40645
Alpha [%]                             4.20602
Beta                                  0.33649
Max. Drawdown [%]                    -5.09853
Avg. Drawdown [%]                     -0.5186
Max. Drawdown Duration        4 days 20:50:00
Avg. Drawdown Duration        0 days 11:13:00
# Trades                          

Backtest.run:   0%|          | 0/2720 [00:00<?, ?bar/s]

Start                     2025-12-28 00:05:00
End                       2026-01-07 00:00:00
Duration                      9 days 23:55:00
Exposure Time [%]                    38.57639
Equity Final [$]                 1011293.0231
Equity Peak [$]                  1017672.6455
Commissions [$]                    17978.6769
Return [%]                             1.1293
Buy & Hold Return [%]                 2.31325
Return (Ann.) [%]                    23.95983
Volatility (Ann.) [%]                21.67773
CAGR [%]                             50.68594
Sharpe Ratio                          1.10527
Sortino Ratio                         1.73808
Calmar Ratio                          5.72794
Alpha [%]                             0.53128
Beta                                  0.25852
Max. Drawdown [%]                    -4.18297
Avg. Drawdown [%]                    -0.93933
Max. Drawdown Duration        7 days 17:15:00
Avg. Drawdown Duration        1 days 00:18:00
# Trades                          

In [5]:
# optimisation for buy and exit quantile using grid search
# shown some common globla optimums that we can sitck to such as window 160 ,buy quanitle 0.01 ,exit quantile 0.7

# --- 2. GRID SEARCH CONFIG ---
# Define ranges to test (Broad enough to find the sweet spot, narrow enough to compute quickly)
window_range = range(40, 161, 20)      # [40, 80, 120, 160]
buy_q_range = [0.01, 0.03, 0.05, 0.10] # Testing up to the bottom 10%
exit_q_range = [0.50, 0.60, 0.70]      # Testing median vs holding for a stronger bounce

days_per_window = 10
bars_per_day = (24 * 60 // 5)
window_size = bars_per_day * days_per_window 

optimization_log = []

# --- 3. MULTI-WINDOW LOOP ---
# Loop BACKWARDS through 10-day blocks
for i in range(len(full_data) - window_size, -1, -window_size):
    window_df = full_data.iloc[i : i + window_size]
    start_label = window_df.index[0].strftime('%Y-%m-%d')
    
    # Initialize Backtest (Using your updated 0.1% commission)
    bt = Backtest(window_df, PercentileStrategy, cash=10000000, commission=0.001)
    
    # Run Grid Search
    stats = bt.optimize(
        window=window_range,
        buy_quantile=buy_q_range,
        exit_quantile=exit_q_range,
        # CONSTRAINT: You cannot have a buy target higher than an exit target
        constraint=lambda p: p.buy_quantile < p.exit_quantile,
        maximize='Sharpe Ratio' 
    )
    
    # Save the "winner" for this window
    optimization_log.append({
        'Window_Start': start_label,
        'Best_Window': stats._strategy.window,
        'Best_Buy_Q': stats._strategy.buy_quantile,
        'Best_Exit_Q': stats._strategy.exit_quantile,
        'Sharpe': stats['Sharpe Ratio'],
        'Return [%]': stats['Return [%]'],
        'Trades': stats['# Trades']
    })

# --- 4. RESULTS SUMMARY ---
opt_df = pd.DataFrame(optimization_log)
print("--- Optimization Results per Window ---")
print(opt_df.to_string())

# Find the "Global Winner" (The most frequent settings)
print("\n--- Most Frequent 'Robust' Settings ---")
print(f"Recommended Window: {opt_df['Best_Window'].mode()[0]}")
print(f"Recommended Buy Quantile: {opt_df['Best_Buy_Q'].mode()[0]}")
print(f"Recommended Exit Quantile: {opt_df['Best_Exit_Q'].mode()[0]}")

/opt/anaconda3/lib/python3.13/site-packages/backtesting/backtesting.py:1624: RuntimeWarning: If you want to use multi-process optimization with `multiprocessing.get_start_method() == 'spawn'` (e.g. on Windows),set `backtesting.Pool = multiprocessing.Pool` (or of the desired context) and hide `bt.optimize()` call behind a `if __name__ == '__main__'` guard. Currently using thread-based paralellism, which might be slightly slower for non-numpy / non-GIL-releasing code. See https://github.com/kernc/backtesting.py/issues/1256
  output = _optimize_grid()


Backtest.optimize:   0%|          | 0/84 [00:00<?, ?it/s]

Backtest.run:   0%|          | 0/2840 [00:00<?, ?bar/s]

/opt/anaconda3/lib/python3.13/site-packages/backtesting/backtesting.py:1637: UserWarning: Some trades remain open at the end of backtest. Use `Backtest(..., finalize_trades=True)` to close them and include them in stats.
  for stats in (bt.run(**params)


Backtest.run:   0%|          | 0/2840 [00:00<?, ?bar/s]

...

Backtest.run:   0%|          | 0/2720 [00:00<?, ?bar/s]

Backtest.run:   0%|          | 0/2720 [00:00<?, ?bar/s]

/opt/anaconda3/lib/python3.13/site-packages/backtesting/backtesting.py:1637: UserWarning: Some trades remain open at the end of backtest. Use `Backtest(..., finalize_trades=True)` to close them and include them in stats.
  for stats in (bt.run(**params)


Backtest.run:   0%|          | 0/2740 [00:00<?, ?bar/s]

--- Optimization Results per Window ---
  Window_Start  Best_Window  Best_Buy_Q  Best_Exit_Q     Sharpe  Return [%]  Trades
0   2026-03-08          100        0.10          0.6   2.011767    2.169044      28
1   2026-02-26          160        0.01          0.7   1.158201    2.063379      12
2   2026-02-16          140        0.01          0.7  -1.437169   -2.317691      12
3   2026-02-06          160        0.03          0.6   8.046261    4.407673      13
4   2026-01-27          160        0.01          0.5 -43.854014  -10.857140      15
5   2026-01-17          100        0.01          0.6   1.207697    1.407368      20
6   2026-01-07          160        0.01          0.7   1.285758    0.797363       9
7   2025-12-28          140        0.01          0.5   6.253560    4.572966      17

--- Most Frequent 'Robust' Settings ---
Recommended Window: 160
Recommended Buy Quantile: 0.01
Recommended Exit Quantile: 0.6


In [21]:
import numpy as np
import pandas as pd
from backtesting import Backtest, Strategy

# --- Helper Functions ---
def get_quantile(series, window, q):
    return series.rolling(window).quantile(q)

def get_fast_hurst(series, window, tau=10):
    series = pd.Series(series)
    log_p = np.log(series)
    var_1 = log_p.diff(1).rolling(window).var()
    var_tau = log_p.diff(tau).rolling(window).var()
    epsilon = 1e-8
    return 0.5 * (np.log((var_tau + epsilon) / (var_1 + epsilon)) / np.log(tau))

# --- Strategy Class (Core Locked, Hurst Unlocked) ---
class BCH_Hurst_Optimizer(Strategy):
    # LOCKED Core Settings
    window = 160
    buy_quantile = 0.01  
    exit_quantile = 0.70 
    
    # UNLOCKED Hurst Settings (Defaults to be overwritten by optimizer)
    hurst_window = 96     
    hurst_threshold = 0.45 

    def init(self):
        close = pd.Series(self.data.Close)
        self.lower_bound = self.I(get_quantile, close, self.window, self.buy_quantile)
        self.exit_bound = self.I(get_quantile, close, self.window, self.exit_quantile)
        self.hurst = self.I(get_fast_hurst, close, self.hurst_window)

    def next(self):
        if len(self.data) < max(self.window, self.hurst_window): return
        
        price = self.data.Close[-1]
        current_hurst = self.hurst[-1]
        if pd.isna(current_hurst): return
        
        is_mean_reverting = current_hurst < self.hurst_threshold
        
        if price <= self.lower_bound[-1] and is_mean_reverting and not self.position:
            self.buy(size=0.9)
            
        elif self.position and price >= self.exit_bound[-1]:
            self.position.close()

In [22]:
import matplotlib.pyplot as plt

# --- CONFIG ---
days_per_window = 10
bars_per_day = (24 * 60 // 5)  # 288 bars for 5m data
window_size = bars_per_day * days_per_window 

all_stats = []

# Loop BACKWARDS: Start at the end, jump back by window_size
# We use a negative step in range()
for i in range(len(full_data) - window_size, -1, -window_size):
    window_df = full_data.iloc[i : i + window_size]
    
    # Label by Start and End date for clarity
    start_label = window_df.index[0].strftime('%Y-%m-%d')
    end_label = window_df.index[-1].strftime('%m-%d')
    label = f"{start_label} to {end_label}"
    
    # Run Backtest
    bt = Backtest(window_df, BCH_Hurst_Optimizer, cash=1000000, commission=0.001)
    stats = bt.run()
    print(stats)
    print(stats['_trades'])
    
    # Store key metrics
    all_stats.append({
        'Window': label,
        'Return [%]': stats['Return [%]'],
        'Sharpe': stats['Sharpe Ratio'],
        'Sortino': stats['Sortino Ratio'],
        'Max DD [%]': stats['Max. Drawdown [%]'],
        'Win Rate [%]': stats['Win Rate [%]'],
        'Profit Factor': stats['Profit Factor'],
        'Trades': stats['# Trades']
    })

# Convert to DataFrame
results_df = pd.DataFrame(all_stats).set_index('Window').fillna(0)

# Display raw table in descending order (Newest first)
print("--- Backtest Results (Reverse Chronological) ---")
print(results_df)

Backtest.run:   0%|          | 0/2720 [00:00<?, ?bar/s]

Start                     2026-03-08 11:20:00
End                       2026-03-18 11:15:00
Duration                      9 days 23:55:00
Exposure Time [%]                    17.15278
Equity Final [$]                 1030247.3224
Equity Peak [$]                  1039553.3224
Commissions [$]                    12867.4776
Return [%]                            3.02473
Buy & Hold Return [%]                 4.69904
Return (Ann.) [%]                    168.7957
Volatility (Ann.) [%]                56.23616
CAGR [%]                            196.84424
Sharpe Ratio                          3.00155
Sortino Ratio                        34.82888
Calmar Ratio                         79.36648
Alpha [%]                             2.49474
Beta                                  0.11279
Max. Drawdown [%]                    -2.12679
Avg. Drawdown [%]                     -0.3259
Max. Drawdown Duration        3 days 01:40:00
Avg. Drawdown Duration        0 days 07:15:00
# Trades                          

/var/folders/f_/xv2kgn592hg5q7v5tl2sgx2m0000gn/T/ipykernel_22127/2550045898.py:22: UserWarning: Some trades remain open at the end of backtest. Use `Backtest(..., finalize_trades=True)` to close them and include them in stats.
  stats = bt.run()


Backtest.run:   0%|          | 0/2720 [00:00<?, ?bar/s]

Start                     2026-02-26 11:20:00
End                       2026-03-08 11:15:00
Duration                      9 days 23:55:00
Exposure Time [%]                        30.0
Equity Final [$]                 1029306.4639
Equity Peak [$]                  1036041.6712
Commissions [$]                    16351.6361
Return [%]                            2.93065
Buy & Hold Return [%]                -5.19235
Return (Ann.) [%]                     160.769
Volatility (Ann.) [%]                92.72543
CAGR [%]                            187.10489
Sharpe Ratio                          1.73382
Sortino Ratio                         8.34548
Calmar Ratio                         24.14552
Alpha [%]                             4.40796
Beta                                  0.28452
Max. Drawdown [%]                    -6.65834
Avg. Drawdown [%]                    -0.73075
Max. Drawdown Duration        5 days 23:05:00
Avg. Drawdown Duration        0 days 12:31:00
# Trades                          

...

Backtest.run:   0%|          | 0/2720 [00:00<?, ?bar/s]

Start                     2026-01-07 11:20:00
End                       2026-01-17 11:15:00
Duration                      9 days 23:55:00
Exposure Time [%]                      21.875
Equity Final [$]                  1031945.803
Equity Peak [$]                  1039006.3475
Commissions [$]                    12796.6212
Return [%]                            3.19458
Buy & Hold Return [%]                -5.35771
Return (Ann.) [%]                   183.89674
Volatility (Ann.) [%]                71.53177
CAGR [%]                            215.24601
Sharpe Ratio                          2.57084
Sortino Ratio                          13.588
Calmar Ratio                         36.96028
Alpha [%]                             4.55043
Beta                                  0.25307
Max. Drawdown [%]                    -4.97552
Avg. Drawdown [%]                    -0.57369
Max. Drawdown Duration        5 days 08:05:00
Avg. Drawdown Duration        0 days 15:13:00
# Trades                          

/var/folders/f_/xv2kgn592hg5q7v5tl2sgx2m0000gn/T/ipykernel_22127/2550045898.py:22: UserWarning: Some trades remain open at the end of backtest. Use `Backtest(..., finalize_trades=True)` to close them and include them in stats.
  stats = bt.run()


Backtest.run:   0%|          | 0/2720 [00:00<?, ?bar/s]

Start                     2025-12-28 11:20:00
End                       2026-01-07 11:15:00
Duration                      9 days 23:55:00
Exposure Time [%]                    30.83333
Equity Final [$]                  996814.1772
Equity Peak [$]                  1009620.3928
Commissions [$]                    12417.6228
Return [%]                           -0.31858
Buy & Hold Return [%]                 1.26806
Return (Ann.) [%]                   -10.04675
Volatility (Ann.) [%]                15.90134
CAGR [%]                            -10.99776
Sharpe Ratio                         -0.63182
Sortino Ratio                        -0.67634
Calmar Ratio                         -2.40235
Alpha [%]                            -0.59645
Beta                                  0.21913
Max. Drawdown [%]                    -4.18206
Avg. Drawdown [%]                    -1.58953
Max. Drawdown Duration        9 days 08:05:00
Avg. Drawdown Duration        3 days 02:54:00
# Trades                          

In [20]:
# improve on multiple entries now strategy enters 3 times according to the percentile range 1 at 0.01 , 2 at 0.005, 3 at 0.001, average entry price 
# below backtest this strat on previous 10 day windows to see performance, not Walk Foward Analysis, WFA in the cell below,
# results shown that averging entry price and entering 3 times instead of one is an improvement 

import numpy as np
import pandas as pd
from backtesting import Backtest, Strategy

def get_quantile(series, window, q):
    """Simple, robust rolling quantile"""
    return series.rolling(window).quantile(q)

class ScaledPercentile_BCH(Strategy):
    # Your proven Global Optimums
    window = 160
    exit_q = 0.70
    
    # The 3 Tranche Levels (The Catching Net)
    q1_buy = 0.010  # 1.0% (The Scout)
    q2_buy = 0.005  # 0.5% (The Core)
    q3_buy = 0.001  # 0.1% (The Sniper)

    def init(self):
        close = pd.Series(self.data.Close)
        
        # Calculate the 3 entry floors and the 1 exit ceiling
        self.b1 = self.I(get_quantile, close, self.window, self.q1_buy)
        self.b2 = self.I(get_quantile, close, self.window, self.q2_buy)
        self.b3 = self.I(get_quantile, close, self.window, self.q3_buy)
        self.exit_target = self.I(get_quantile, close, self.window, self.exit_q)

    def next(self):
        if len(self.data) < self.window: return
        
        price = self.data.Close[-1]
        open_trades = len(self.trades)
        
        # --- SCALING IN (ENTRY LOGIC) ---
        # Bullet 1: Fires at the 1% extreme
        if open_trades == 0 and price <= self.b1[-1]:
            # size=0.3 uses 30% of REMAINING cash. 
            self.buy(size=0.2) 
            
        # Bullet 2: Fires if the crash deepens to the 0.5% extreme
        elif open_trades == 1 and price <= self.b2[-1]:
            # Uses 30% of what's left (Risk naturally decreases as the knife falls)
            self.buy(size=0.375)
            
        # Bullet 3: Fires at the absolute 0.1% panic extreme
        elif open_trades == 2 and price <= self.b3[-1]:
            self.buy(size=0.8)
            
        # --- EXIT LOGIC ---
        # If we hold ANY positions, and price hits the 70th percentile bounce, close everything
        elif open_trades > 0 and price >= self.exit_target[-1]:
            self.position.close()
            
# --- CONFIG ---
days_per_window = 10
bars_per_day = (24 * 60 // 5)  # 288 bars for 5m data
window_size = bars_per_day * days_per_window 

all_stats = []

# Loop BACKWARDS: Start at the end, jump back by window_size
# We use a negative step in range()
for i in range(len(full_data) - window_size, -1, -window_size):
    window_df = full_data.iloc[i : i + window_size]
    
    # Label by Start and End date for clarity
    start_label = window_df.index[0].strftime('%Y-%m-%d')
    end_label = window_df.index[-1].strftime('%m-%d')
    label = f"{start_label} to {end_label}"
    
    # Run Backtest
    bt = Backtest(window_df, ScaledPercentile_BCH, cash=1000000, commission=0.001)
    stats = bt.run()
    print(stats)
    print(stats['_trades'])
    
    # Store key metrics
    all_stats.append({
        'Window': label,
        'Return [%]': stats['Return [%]'],
        'Sharpe': stats['Sharpe Ratio'],
        'Sortino': stats['Sortino Ratio'],
        'Max DD [%]': stats['Max. Drawdown [%]'],
        'Win Rate [%]': stats['Win Rate [%]'],
        'Profit Factor': stats['Profit Factor'],
        'Trades': stats['# Trades']
    })

# Convert to DataFrame
results_df = pd.DataFrame(all_stats).set_index('Window').fillna(0)

# Display raw table in descending order (Newest first)
print("--- Backtest Results (Reverse Chronological) ---")
print(results_df)

Backtest.run:   0%|          | 0/2720 [00:00<?, ?bar/s]

Start                     2026-03-08 00:05:00
End                       2026-03-18 00:00:00
Duration                      9 days 23:55:00
Exposure Time [%]                    29.23611
Equity Final [$]                 1034867.9169
Equity Peak [$]                  1034995.9623
Commissions [$]                    14057.5775
Return [%]                            3.48679
Buy & Hold Return [%]                 5.64516
Return (Ann.) [%]                   276.59937
Volatility (Ann.) [%]                44.36939
CAGR [%]                            249.53318
Sharpe Ratio                          6.23401
Sortino Ratio                        98.28923
Calmar Ratio                        138.70916
Alpha [%]                             2.53094
Beta                                  0.16932
Max. Drawdown [%]                     -1.9941
Avg. Drawdown [%]                    -0.21924
Max. Drawdown Duration        2 days 03:50:00
Avg. Drawdown Duration        0 days 05:13:00
# Trades                          

/var/folders/f_/xv2kgn592hg5q7v5tl2sgx2m0000gn/T/ipykernel_59944/213974306.py:73: UserWarning: Some trades remain open at the end of backtest. Use `Backtest(..., finalize_trades=True)` to close them and include them in stats.
  stats = bt.run()


Backtest.run:   0%|          | 0/2720 [00:00<?, ?bar/s]

Start                     2026-02-26 00:05:00
End                       2026-03-08 00:00:00
Duration                      9 days 23:55:00
Exposure Time [%]                    41.45833
Equity Final [$]                  966464.4949
Equity Peak [$]                  1001772.8503
Commissions [$]                    13446.6067
Return [%]                           -3.35355
Buy & Hold Return [%]                -9.64508
Return (Ann.) [%]                   -27.82847
Volatility (Ann.) [%]                22.16755
CAGR [%]                            -71.21954
Sharpe Ratio                         -1.25537
Sortino Ratio                        -1.30294
Calmar Ratio                         -2.95484
Alpha [%]                             -0.1157
Beta                                   0.3357
Max. Drawdown [%]                    -9.41793
Avg. Drawdown [%]                    -3.18518
Max. Drawdown Duration        9 days 09:05:00
Avg. Drawdown Duration        3 days 03:17:00
# Trades                          

...

Start                     2026-01-17 00:05:00
End                       2026-01-27 00:00:00
Duration                      9 days 23:55:00
Exposure Time [%]                    38.95833
Equity Final [$]                    1020658.0
Equity Peak [$]                  1042410.1108
Commissions [$]                       17122.0
Return [%]                             2.0658
Buy & Hold Return [%]                -3.21608
Return (Ann.) [%]                    84.84759
Volatility (Ann.) [%]                56.77244
CAGR [%]                             110.9801
Sharpe Ratio                          1.49452
Sortino Ratio                         3.90009
Calmar Ratio                         19.66646
Alpha [%]                             3.21164
Beta                                  0.35628
Max. Drawdown [%]                    -4.31433
Avg. Drawdown [%]                     -0.5023
Max. Drawdown Duration        3 days 08:25:00
Avg. Drawdown Duration        0 days 06:58:00
# Trades                          

Backtest.run:   0%|          | 0/2720 [00:00<?, ?bar/s]

Start                     2026-01-07 00:05:00
End                       2026-01-17 00:00:00
Duration                      9 days 23:55:00
Exposure Time [%]                    31.21528
Equity Final [$]                 1033257.0004
Equity Peak [$]                   1047671.154
Commissions [$]                    16536.1996
Return [%]                             3.3257
Buy & Hold Return [%]                -5.21492
Return (Ann.) [%]                     98.6805
Volatility (Ann.) [%]                36.90087
CAGR [%]                            230.20614
Sharpe Ratio                          2.67421
Sortino Ratio                         8.09521
Calmar Ratio                         23.11585
Alpha [%]                             4.92069
Beta                                  0.30585
Max. Drawdown [%]                    -4.26895
Avg. Drawdown [%]                    -0.55768
Max. Drawdown Duration        3 days 07:05:00
Avg. Drawdown Duration        0 days 10:11:00
# Trades                          

Backtest.run:   0%|          | 0/2720 [00:00<?, ?bar/s]

Start                     2025-12-28 00:05:00
End                       2026-01-07 00:00:00
Duration                      9 days 23:55:00
Exposure Time [%]                    38.57639
Equity Final [$]                 1007777.0428
Equity Peak [$]                  1020280.6485
Commissions [$]                    15189.7572
Return [%]                             0.7777
Buy & Hold Return [%]                 2.31325
Return (Ann.) [%]                     3.58899
Volatility (Ann.) [%]                16.00129
CAGR [%]                             32.69223
Sharpe Ratio                          0.22429
Sortino Ratio                         0.28601
Calmar Ratio                          0.85785
Alpha [%]                             0.22898
Beta                                  0.23721
Max. Drawdown [%]                    -4.18368
Avg. Drawdown [%]                    -0.66639
Max. Drawdown Duration        8 days 20:50:00
Avg. Drawdown Duration        1 days 00:17:00
# Trades                          

In [21]:
# Walk Forward Analysis with ScaledPercentile_BCH

import pandas as pd
from backtesting import Backtest, Strategy

# --- 2. Walk-Forward Configuration ---
# 20 days train, 10 days test
train_days = 20
test_days = 10
bars_per_day = (24 * 60 // 5)

train_size = bars_per_day * train_days
test_size = bars_per_day * test_days

# Parameter Grid (Based on your best BCH results)
w_range = [100, 120, 140, 160, 180, 200]
q1_range = [0.010, 0.020, 0.030]  
q2_range = [0.005, 0.010]         
q3_range = [0.001, 0.005]        
eq_range = [0.60, 0.70, 0.80]

walk_forward_log = []

# --- 3. Walk-Forward Loop ---
print("Starting Walk-Forward Analysis. This may take a moment...")

# Loop starts from the beginning of the dataset
for i in range(0, len(full_data) - train_size - test_size, test_size):
    
    # Define Train/Test Data splits
    train_df = full_data.iloc[i : i + train_size]
    test_df = full_data.iloc[i + train_size : i + train_size + test_size]
    
    test_start_label = test_df.index[0].strftime('%Y-%m-%d')
    test_end_label = test_df.index[-1].strftime('%Y-%m-%d')
    
    # --- TRAINING PHASE (In-Sample) ---
    bt_train = Backtest(train_df, ScaledPercentile_BCH, cash=10000000, commission=0.001)
    train_stats = bt_train.optimize(
        window=w_range,
        q1_buy=q1_range,
        q2_buy=q2_range,
        q3_buy=q3_range,
        exit_q=eq_range,
        constraint=lambda p: (p.q3_buy < p.q2_buy < p.q1_buy) and (p.q1_buy < p.exit_q),
        maximize='Sharpe Ratio'
    )
    
    # Extract winning parameters
    best_w = train_stats._strategy.window
    best_q1 = train_stats._strategy.q1_buy
    best_q2 = train_stats._strategy.q2_buy
    best_q3 = train_stats._strategy.q3_buy
    best_eq = train_stats._strategy.exit_q
    
    # --- TESTING PHASE (Out-of-Sample) ---
    # Create a dynamic class with the locked training parameters
    class LiveStrategy(ScaledPercentile_BCH):
        window = best_w
        q1_buy = best_q1
        q2_buy = best_q2
        q3_buy = best_q3
        exit_q = best_eq
        
    bt_test = Backtest(test_df, LiveStrategy, cash=10000000, commission=0.001)
    test_stats = bt_test.run()
    print(test_stats)
    print(test_stats['_trades'])
    
    # Record OOS Results
    walk_forward_log.append({
        'Test_Window': f"{test_start_label} to {test_end_label}",
        'Locked_Win': best_w,
        'Locked_Q1': best_q1,
        'Locked_Q2': best_q2,
        'Locked_Q3': best_q3,
        'Locked_EQ': best_eq,
        'OOS_Sharpe': test_stats['Sharpe Ratio'],
        'OOS_Return [%]': test_stats['Return [%]'],
        'OOS_Max_DD [%]': test_stats['Max. Drawdown [%]'],
        'Trades': test_stats['# Trades']
    })

# --- 4. Final Output ---
wfa_df = pd.DataFrame(walk_forward_log)
print("\n--- TRUE OUT-OF-SAMPLE (WALK-FORWARD) RESULTS ---")
print(wfa_df.to_string())

net_return = wfa_df['OOS_Return [%]'].sum()
print(f"\nNet Out-of-Sample Return: {net_return:.2f}%")

Starting Walk-Forward Analysis. This may take a moment...


/opt/anaconda3/lib/python3.13/site-packages/backtesting/backtesting.py:1624: RuntimeWarning: If you want to use multi-process optimization with `multiprocessing.get_start_method() == 'spawn'` (e.g. on Windows),set `backtesting.Pool = multiprocessing.Pool` (or of the desired context) and hide `bt.optimize()` call behind a `if __name__ == '__main__'` guard. Currently using thread-based paralellism, which might be slightly slower for non-numpy / non-GIL-releasing code. See https://github.com/kernc/backtesting.py/issues/1256
  output = _optimize_grid()


Backtest.optimize:   0%|          | 0/126 [00:00<?, ?it/s]

Backtest.run:   0%|          | 0/5660 [00:00<?, ?bar/s]

Backtest.run:   0%|          | 0/5660 [00:00<?, ?bar/s]

...

Backtest.run:   0%|          | 0/5560 [00:00<?, ?bar/s]

Backtest.run:   0%|          | 0/5560 [00:00<?, ?bar/s]

Backtest.run:   0%|          | 0/5600 [00:00<?, ?bar/s]

Backtest.run:   0%|          | 0/2720 [00:00<?, ?bar/s]

Start                     2026-03-05 00:00:00
End                       2026-03-14 23:55:00
Duration                      9 days 23:55:00
Exposure Time [%]                    31.07639
Equity Final [$]                10329983.1034
Equity Peak [$]                 10436175.5936
Commissions [$]                   173051.0966
Return [%]                            3.29983
Buy & Hold Return [%]                 1.16833
Return (Ann.) [%]                   127.57409
Volatility (Ann.) [%]                40.43521
CAGR [%]                            227.20095
Sharpe Ratio                          3.15502
Sortino Ratio                        14.69907
Calmar Ratio                          41.8969
Alpha [%]                              3.0865
Beta                                   0.1826
Max. Drawdown [%]                    -3.04495
Avg. Drawdown [%]                    -0.26306
Max. Drawdown Duration        2 days 12:00:00
Avg. Drawdown Duration        0 days 05:40:00
# Trades                          

In [26]:
# add hurst exponent to determine mean reversion / trending act as the main switch best strat as of now 
# below Walk Foward Analysis for the results , shown better results

import numpy as np
import pandas as pd
from backtesting import Backtest, Strategy

# --- 1. The Regime Filter ---
def fast_rolling_hurst(price, window):
    """
    Fast Variance Ratio approximation of the Hurst Exponent.
    H < 0.5 = Mean Reverting | H >= 0.5 = Trending
    """
    lp = np.log(price)
    lag1, lag2 = 2, 20
    
    var1 = (lp - lp.shift(lag1)).rolling(window).var().replace(0, 1e-8)
    var2 = (lp - lp.shift(lag2)).rolling(window).var().replace(0, 1e-8)
    
    return 0.5 * np.log(var2 / var1) / np.log(lag2 / lag1)

def get_quantile(series, window, q):
    return series.rolling(window).quantile(q)

# --- 2. The Core Strategy ---
class ScaledPercentile_Hurst_BCH(Strategy):
    window = 160
    
    # Entry Tranches
    q1_buy = 0.010  
    q2_buy = 0.005  
    q3_buy = 0.001  
    
    # Exit Target
    exit_q = 0.70   
    
    # Locked Regime Filter (Requires strong mean-reversion)
    hurst_threshold = 0.45 

    def init(self):
        close = pd.Series(self.data.Close)
        
        self.b1 = self.I(get_quantile, close, self.window, self.q1_buy)
        self.b2 = self.I(get_quantile, close, self.window, self.q2_buy)
        self.b3 = self.I(get_quantile, close, self.window, self.q3_buy)
        self.exit_target = self.I(get_quantile, close, self.window, self.exit_q)
        
        # Initialize the Hurst sensor
        self.hurst = self.I(fast_rolling_hurst, close, self.window)

    def next(self):
        if len(self.data) < self.window: return
        
        price = self.data.Close[-1]
        open_trades = len(self.trades)
        h_val = self.hurst[-1]
        
        if pd.isna(h_val): return
        
        # --- EXIT LOGIC ---
        if open_trades > 0 and price >= self.exit_target[-1]:
            self.position.close()
            return 
            
        # --- THE PADLOCK (Regime Filter) ---
        # If the 13-hour trend is directional, block all new entries
        if h_val >= self.hurst_threshold:
            return
            
        # --- PYRAMID ENTRY LOGIC ---
        # 20% / 30% / 40% of Total Starting Capital
        if open_trades == 0 and price <= self.b1[-1]:
            self.buy(size=0.20) 
        elif open_trades == 1 and price <= self.b2[-1]:
            self.buy(size=0.375)
        elif open_trades == 2 and price <= self.b3[-1]:
            self.buy(size=0.80)

# --- 3. Walk-Forward Configuration ---
train_days = 20
test_days = 10
bars_per_day = (24 * 60 // 5)

train_size = bars_per_day * train_days
test_size = bars_per_day * test_days

w_range = [120, 140, 160]
q1_range = [0.01, 0.02, 0.03]  
q2_range = [0.005, 0.01]         
q3_range = [0.001, 0.005]        
eq_range = [0.60, 0.70, 0.80]

walk_forward_log = []

# --- 4. Walk-Forward Loop ---
print("Starting WFA: Pyramid Sizing + Hurst Regime Filter...")

for i in range(0, len(full_data) - train_size - test_size, test_size):
    
    train_df = full_data.iloc[i : i + train_size]
    test_df = full_data.iloc[i + train_size : i + train_size + test_size]
    
    test_start_label = test_df.index[0].strftime('%Y-%m-%d')
    test_end_label = test_df.index[-1].strftime('%Y-%m-%d')
    
    bt_train = Backtest(train_df, ScaledPercentile_Hurst_BCH, cash=10000000, commission=0.001)
    
    train_stats = bt_train.optimize(
        window=w_range,
        q1_buy=q1_range,
        q2_buy=q2_range,
        q3_buy=q3_range,
        exit_q=eq_range,
        constraint=lambda p: (p.q3_buy < p.q2_buy < p.q1_buy) and (p.q1_buy < p.exit_q),
        maximize='Sharpe Ratio'
    )
    
    best_w = train_stats._strategy.window
    best_q1 = train_stats._strategy.q1_buy
    best_q2 = train_stats._strategy.q2_buy
    best_q3 = train_stats._strategy.q3_buy
    best_eq = train_stats._strategy.exit_q
    
    class LiveStrategy(ScaledPercentile_Hurst_BCH):
        window = best_w
        q1_buy = best_q1
        q2_buy = best_q2
        q3_buy = best_q3
        exit_q = best_eq
        # hurst_threshold remains statically locked at 0.45
        
    bt_test = Backtest(test_df, LiveStrategy, cash=10000000, commission=0.001)
    test_stats = bt_test.run()
    print(test_stats)
    print(test_stats['_trades'])
    
    walk_forward_log.append({
        'Test_Window': f"{test_start_label} to {test_end_label}",
        'Win': best_w,
        'Q1/Q2/Q3': f"{best_q1}/{best_q2}/{best_q3}",
        'EQ': best_eq,
        'OOS_Sharpe': test_stats['Sharpe Ratio'],
        'OOS_Return [%]': test_stats['Return [%]'],
        'OOS_Max_DD [%]': test_stats['Max. Drawdown [%]'],
        'Trades': test_stats['# Trades']
    })

wfa_df = pd.DataFrame(walk_forward_log)
print("\n--- TRUE OUT-OF-SAMPLE (WALK-FORWARD) RESULTS ---")
print(wfa_df.to_string())

net_return = wfa_df['OOS_Return [%]'].sum()
print(f"\nNet Out-of-Sample Return: {net_return:.2f}%")

Starting WFA: Pyramid Sizing + Hurst Regime Filter...


/opt/anaconda3/lib/python3.13/site-packages/backtesting/backtesting.py:1624: RuntimeWarning: If you want to use multi-process optimization with `multiprocessing.get_start_method() == 'spawn'` (e.g. on Windows),set `backtesting.Pool = multiprocessing.Pool` (or of the desired context) and hide `bt.optimize()` call behind a `if __name__ == '__main__'` guard. Currently using thread-based paralellism, which might be slightly slower for non-numpy / non-GIL-releasing code. See https://github.com/kernc/backtesting.py/issues/1256
  output = _optimize_grid()


Backtest.optimize:   0%|          | 0/63 [00:00<?, ?it/s]

Backtest.run:   0%|          | 0/5620 [00:00<?, ?bar/s]

Backtest.run:   0%|          | 0/5620 [00:00<?, ?bar/s]

...

Backtest.run:   0%|          | 0/5580 [00:00<?, ?bar/s]

Backtest.run:   0%|          | 0/5580 [00:00<?, ?bar/s]

Backtest.run:   0%|          | 0/5620 [00:00<?, ?bar/s]

Backtest.run:   0%|          | 0/2740 [00:00<?, ?bar/s]

Start                     2026-03-05 00:00:00
End                       2026-03-14 23:55:00
Duration                      9 days 23:55:00
Exposure Time [%]                    38.57639
Equity Final [$]                10287121.2876
Equity Peak [$]                  10353813.517
Commissions [$]                   155850.9124
Return [%]                            2.87121
Buy & Hold Return [%]                 1.23403
Return (Ann.) [%]                      137.65
Volatility (Ann.) [%]                39.62167
CAGR [%]                            181.11354
Sharpe Ratio                          3.47411
Sortino Ratio                        18.66189
Calmar Ratio                          55.9243
Alpha [%]                             2.65711
Beta                                   0.1735
Max. Drawdown [%]                    -2.46136
Avg. Drawdown [%]                    -0.23285
Max. Drawdown Duration        2 days 14:20:00
Avg. Drawdown Duration        0 days 05:24:00
# Trades                          

In [42]:
# add a new exit check hurst_panic and check for downward momentum exit if hurst > 0.55 and trend is downwards
# tried Walk Forward Analysis below to see how this strat works performed worse then with the hurst exponent only(ScaledPercentile_Hurst_BCH)

import numpy as np
import pandas as pd
from backtesting import Backtest, Strategy

# --- 1. The Regime Filter ---
def fast_rolling_hurst(price, window):
    """
    Fast Variance Ratio approximation of the Hurst Exponent.
    H < 0.5 = Mean Reverting | H >= 0.5 = Trending
    """
    lp = np.log(price)
    lag1, lag2 = 2, 20
    
    var1 = (lp - lp.shift(lag1)).rolling(window).var().replace(0, 1e-8)
    var2 = (lp - lp.shift(lag2)).rolling(window).var().replace(0, 1e-8)
    
    return 0.5 * np.log(var2 / var1) / np.log(lag2 / lag1)

def get_quantile(series, window, q):
    return series.rolling(window).quantile(q)

# --- 2. The Core Strategy ---
class ScaledPercentile_HurstSwitch_BCH(Strategy):
    window = 160
    
    # Entry Tranches
    q1_buy = 0.010  
    q2_buy = 0.005  
    q3_buy = 0.001  
    
    # Exit Target
    exit_q = 0.70   
    
    # Locked Regime Filter (Requires strong mean-reversion)
    hurst_threshold = 0.45 

    def init(self):
        close = pd.Series(self.data.Close)
        
        self.b1 = self.I(get_quantile, close, self.window, self.q1_buy)
        self.b2 = self.I(get_quantile, close, self.window, self.q2_buy)
        self.b3 = self.I(get_quantile, close, self.window, self.q3_buy)
        self.exit_target = self.I(get_quantile, close, self.window, self.exit_q)
        
        # Initialize the Hurst sensor
        self.hurst = self.I(fast_rolling_hurst, close, self.window)

    # New Optimizer Variable for the Panic Switch
    hurst_panic = 0.55 

    def next(self):
        if len(self.data) < self.window: return
        
        price = self.data.Close[-1]
        open_trades = len(self.trades)
        h_val = self.hurst[-1]
        
        if pd.isna(h_val): return
        
        # --- 1. STANDARD EXIT LOGIC ---
        if open_trades > 0 and price >= self.exit_target[-1]:
            self.position.close()
            return 
            
        # --- 2. THE DIRECTIONAL HURST BAILOUT (New!) ---
        # If we are in a trade and a strong trend suddenly forms...
        if open_trades > 0 and h_val >= self.hurst_panic:
            
            # Check direction: Are we bleeding? 
            # (trades[0] is our first, highest entry. If price is below it, we are losing)
            if price < self.trades[0].entry_price:
                self.position.close()  # Cut the steamroller
                return
            
            # If price > trades[0].entry_price, it's a V-bottom rocket. 
            # We skip the close() and let it ride!
            
        # --- 3. THE HURST PADLOCK ---
        # Blocks new entries if a trend is already happening
        if h_val >= self.hurst_threshold: 
            return
            
        # --- 4. CASH-DRAG ENTRY LOGIC ---
        if open_trades == 0 and price <= self.b1[-1]:
            self.buy(size=0.2) 
        elif open_trades == 1 and price <= self.b2[-1]:
            self.buy(size=0.375)
        elif open_trades == 2 and price <= self.b3[-1]:
            self.buy(size=0.8)

# --- 3. Walk-Forward Configuration ---
train_days = 20
test_days = 10
bars_per_day = (24 * 60 // 5)

train_size = bars_per_day * train_days
test_size = bars_per_day * test_days

w_range = [120, 140, 160]
q1_range = [0.01, 0.02, 0.03]  
q2_range = [0.005, 0.01]         
q3_range = [0.001, 0.005]        
eq_range = [0.60, 0.70, 0.80]

walk_forward_log = []

# --- 4. Walk-Forward Loop ---
print("Starting WFA: Pyramid Sizing + Hurst Regime Filter...")

for i in range(0, len(full_data) - train_size - test_size, test_size):
    
    train_df = full_data.iloc[i : i + train_size]
    test_df = full_data.iloc[i + train_size : i + train_size + test_size]
    
    test_start_label = test_df.index[0].strftime('%Y-%m-%d')
    test_end_label = test_df.index[-1].strftime('%Y-%m-%d')
    
    bt_train = Backtest(train_df, ScaledPercentile_HurstSwitch_BCH, cash=10000000, commission=0.001)
    
    train_stats = bt_train.optimize(
        window=w_range,
        q1_buy=q1_range,
        q2_buy=q2_range,
        q3_buy=q3_range,
        exit_q=eq_range,
        constraint=lambda p: (p.q3_buy < p.q2_buy < p.q1_buy) and (p.q1_buy < p.exit_q),
        maximize='Sharpe Ratio'
    )
    
    best_w = train_stats._strategy.window
    best_q1 = train_stats._strategy.q1_buy
    best_q2 = train_stats._strategy.q2_buy
    best_q3 = train_stats._strategy.q3_buy
    best_eq = train_stats._strategy.exit_q
    
    class LiveStrategy(ScaledPercentile_Hurst_BCH):
        window = best_w
        q1_buy = best_q1
        q2_buy = best_q2
        q3_buy = best_q3
        exit_q = best_eq
        # hurst_threshold remains statically locked at 0.45
        
    bt_test = Backtest(test_df, LiveStrategy, cash=10000000, commission=0.001)
    test_stats = bt_test.run()
    print(test_stats)
    print(test_stats['_trades'])
    bt_test.plot()
    
    walk_forward_log.append({
        'Test_Window': f"{test_start_label} to {test_end_label}",
        'Win': best_w,
        'Q1/Q2/Q3': f"{best_q1}/{best_q2}/{best_q3}",
        'EQ': best_eq,
        'OOS_Sharpe': test_stats['Sharpe Ratio'],
        'OOS_Return [%]': test_stats['Return [%]'],
        'OOS_Max_DD [%]': test_stats['Max. Drawdown [%]'],
        'Trades': test_stats['# Trades']
    })

wfa_df = pd.DataFrame(walk_forward_log)
print("\n--- TRUE OUT-OF-SAMPLE (WALK-FORWARD) RESULTS ---")
print(wfa_df.to_string())

net_return = wfa_df['OOS_Return [%]'].sum()
print(f"\nNet Out-of-Sample Return: {net_return:.2f}%")

Starting WFA: Pyramid Sizing + Hurst Regime Filter...


/opt/anaconda3/lib/python3.13/site-packages/backtesting/backtesting.py:1624: RuntimeWarning: If you want to use multi-process optimization with `multiprocessing.get_start_method() == 'spawn'` (e.g. on Windows),set `backtesting.Pool = multiprocessing.Pool` (or of the desired context) and hide `bt.optimize()` call behind a `if __name__ == '__main__'` guard. Currently using thread-based paralellism, which might be slightly slower for non-numpy / non-GIL-releasing code. See https://github.com/kernc/backtesting.py/issues/1256
  output = _optimize_grid()


Backtest.optimize:   0%|          | 0/63 [00:00<?, ?it/s]

Backtest.run:   0%|          | 0/5620 [00:00<?, ?bar/s]

Backtest.run:   0%|          | 0/5620 [00:00<?, ?bar/s]

...

Backtest.run:   0%|          | 0/5580 [00:00<?, ?bar/s]

Backtest.run:   0%|          | 0/5580 [00:00<?, ?bar/s]

Backtest.run:   0%|          | 0/2700 [00:00<?, ?bar/s]

Start                     2026-03-05 00:00:00
End                       2026-03-14 23:55:00
Duration                      9 days 23:55:00
Exposure Time [%]                    32.74306
Equity Final [$]                10307865.9811
Equity Peak [$]                 10336669.1401
Commissions [$]                   123070.9189
Return [%]                            3.07866
Buy & Hold Return [%]                 0.79759
Return (Ann.) [%]                   139.74786
Volatility (Ann.) [%]                41.34009
CAGR [%]                            202.57066
Sharpe Ratio                          3.38044
Sortino Ratio                        19.59832
Calmar Ratio                         62.82465
Alpha [%]                             2.92901
Beta                                  0.18763
Max. Drawdown [%]                    -2.22441
Avg. Drawdown [%]                    -0.26362
Max. Drawdown Duration        2 days 03:30:00
Avg. Drawdown Duration        0 days 06:11:00
# Trades                          


--- TRUE OUT-OF-SAMPLE (WALK-FORWARD) RESULTS ---
                Test_Window  Win          Q1/Q2/Q3   EQ  OOS_Sharpe  OOS_Return [%]  OOS_Max_DD [%]  Trades
0  2026-01-14 to 2026-01-23  120  0.01/0.005/0.001  0.6    4.244706        3.224165       -2.624741      37
1  2026-01-24 to 2026-02-02  120  0.02/0.005/0.001  0.8  -43.680720       -8.111682      -13.161272      22
2  2026-02-03 to 2026-02-12  140  0.03/0.005/0.001  0.6   -1.504335       -0.286804       -9.672960      29
3  2026-02-13 to 2026-02-22  120  0.01/0.005/0.001  0.6   10.404218        1.945552       -1.647598      16
4  2026-02-23 to 2026-03-04  120  0.01/0.005/0.001  0.7  -10.983824       -8.197986      -10.780403      20
5  2026-03-05 to 2026-03-14  160   0.02/0.01/0.001  0.8    3.380444        3.078660       -2.224411      26

Net Out-of-Sample Return: -8.35%
